In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ioai-philippines-2026-round-2-semi-finals/ioai_qc_train.csv
/kaggle/input/ioai-philippines-2026-round-2-semi-finals/SUBMISSION_FINAL_INT.csv
/kaggle/input/ioai-philippines-2026-round-2-semi-finals/TEST_FINAL_INT.csv


In [ ]:
# 1. SETUP & DATA LOADING
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# Load the datasets
train = pd.read_csv('/kaggle/input/ioai-qc-rainfall/ioai_qc_train.csv')
test = pd.read_csv('/kaggle/input/ioai-qc-rainfall/TEST_FINAL_INT.csv')
sample_sub = pd.read_csv('/kaggle/input/ioai-qc-rainfall/SUBMISSION_FINAL_INT.csv')

# 2. DATA CLEANING (Syllabus: Handling Sensor Noise)
# Example: Fix the 999% humidity sensor error mentioned in the syllabus
train['humidity'] = train['humidity'].apply(lambda x: np.nan if x > 100 else x)
test['humidity'] = test['humidity'].apply(lambda x: np.nan if x > 100 else x)

# Fill missing values with the median
train = train.fillna(train.median(numeric_only=True))
test = test.fillna(test.median(numeric_only=True))

# 3. FEATURE ENGINEERING (Time-Series Patterns)
def create_features(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['hour'] = df['datetime'].dt.hour
    df['day_of_week'] = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month
    return df

train = create_features(train)
test = create_features(test)

# 4. MODEL TRAINING
# Define features and target
features = ['temp', 'humidity', 'pressure', 'wind_speed', 'hour', 'day_of_week', 'month']
X_train = train[features]
y_train = train['precipitation']

# Simple XGBoost Baseline
model = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# 5. PREDICTION & SUBMISSION (CRITICAL: Using the 'id' column)
# Generate predictions for 2023
test_preds = model.predict(test[features])

# Ensure predictions are not negative (rainfall can't be negative)
test_preds = np.maximum(0, test_preds)

# Create the final submission file
# We use the 'id' column exactly as provided in the test set
submission = pd.DataFrame({
    'id': test['id'],
    'precipitation': test_preds
})

# Save to CSV
submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' created successfully with integer IDs!")